# Lab 05 — Trust zones, and prompt injection as authority confusion

**PART II — Foundations of executable governance**  
*Ch. 6 — Trust Zones and Boundaries*

`intermediate` · about 25 minutes

## By the end of this lab you will be able to

- Explain how a trust zone differs from a network segment or a process boundary
- Model a context item by origin, authority, and taint
- Apply share allowlists and never-share categories
- Analyse prompt injection as authority confusion rather than a model defect

**Concepts:** `trust zone`, `zone membership`, `share allowlist`, `never-share categories`, `context authority`, `taint`, `prompt injection`, `ingress and egress gates`

---

Run the cell below. It executes the *same* `lab.py` the CLI runs — this notebook is a second view onto one implementation, not a copy, so the two can never disagree.


In [1]:
# Make the repository importable from anywhere under notebooks/
import sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / 'pyproject.toml').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

from nornyx_lab.engine import find_lab, run_lab
meta = find_lab('05')
print(meta.title)

Trust zones, and prompt injection as authority confusion


## Run the lab


In [2]:
ctx = run_lab(find_lab('05'))

  Lab 05    Trust zones, and prompt injection as authority confusion

  Textbook: Ch. 6 — Trust Zones and Boundaries

───────────────────────────────────────────────────────────────────────────────────────────────────────────────────

▸ A zone is a declared governance boundary

╭──────────────────────────────────────── concept · not a network segment ────────────────────────────────────────╮
│ A trust zone is declared, not inherited from infrastructure. Two agents in the same Kubernetes namespace, the   │
│ same process, and the same VPC can sit in different zones — and two agents on different continents can sit in   │
│ one.                                                                                                            │
│                                                                                                                 │
│ What a zone carries that a subnet does not:                                                                     │
│                                                                                                                 │
│  • membership — which identities are in it, holding what, until when                                            │
│  • allowed transitions — which other zones it may reach at all                                                  │
│  • a share allowlist — what categories of content may cross                                                     │
│  • never-share categories — what may never leave, under any approval                                            │
│  • ingress and egress gates — where a crossing is evaluated                                                     │
│                                                                                                                 │
│ never_share must be non-empty. A boundary with nothing it refuses to emit is not a boundary; it is a label.     │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

Model every context item on three axes:                                                         

                                                                                        
 item                    origin                                 authority     taint     
 ────────────────────────────────────────────────────────────────────────────────────── 
 network.nyx             the repository, reviewed               decides       trusted   
 docs/refund_policy.md   the repository, reviewed               decides       trusted   
 the retrieved web page  the open internet                      informs only  untrusted 
 the user's prompt       a human, unauthenticated to the agent  informs only  untrusted 
                                                                                        

Prompt injection is not a model defect to be patched. It is authority confusion: text that      
should only inform arrives in the same channel as text that decides, and nothing in the channel 
marks which is which.                                                                           

The fix is not "make the model resist better". A model that resists 99% of injections is a model
that fails on the attempt that matters. The fix is that the action the injected text asks for   
must cross a boundary that never asked the model's opinion.

  include is readability; authority is decision rights

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ contexts:                                                                                                       │
│   - name: ResearchContext                                                                                       │
│     include: [README.md, briefings/**/*.md]   # what may be READ                                                │
│     exclude: [.env, secrets/**]               # what may never be read                                          │
│     authority: [network.nyx]                  # what may DECIDE                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── ⊘ where this stops ───────────────────────────────────────────────╮
│ The zone stopped the crossing. It did not stop the injection.                                                   │
│                                                                                                                 │
│ The planner still read the hostile text, still believed it, and still proposed publishing. Nothing here cleans  │
│ input or improves the model. Everything here assumes the model will be fooled and makes that survivable.        │
│                                                                                                                 │
│ And note the coverage question underneath: this held because the publish path went through the evaluated        │
│ crossing. A second path to the same effect — an HTTP client the agent can call directly — is not covered by     │
│ anything you have seen so far. That is Lab 13.                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── ⚑ your turn ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  1 Add briefing_draft to zone.public_web's share_allowlist, rebuild with python scripts/build_contracts.py      │
│    atlas, and re-run. Which decision changed?                                                                   │
│  2 Now try adding secrets to the same allowlist. Rebuild and re-run. Why does the decision not change — and     │
│    which line in the contract is responsible?                                                                   │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## Inspect what the lab measured

Every lab publishes its findings with `ctx.record(...)`. This is the same data `checks.py` asserts on — poke at it.


In [ ]:
import json
print(json.dumps(ctx.results, indent=2, default=str))

## Prove it

The concept checks for this lab. Each one is a proposition written so a machine can settle it.


In [ ]:
import subprocess, sys
checks = ROOT / 'labs' / '05_trust_zones' / 'checks.py'
proc = subprocess.run(
    [sys.executable, '-m', 'pytest', str(checks), '-v', '--no-header'],
    cwd=str(ROOT), capture_output=True, text=True,
    encoding='utf-8', errors='replace',
)
print(proc.stdout[-4000:])

## Your turn

The lab printed a **your turn** panel above. Do it here — edit the contract, re-run the cells, and watch which decision changes.

---

Next: [Lab 06 — Policy semantics and deterministic evaluation](./06_policy_semantics.ipynb)


In [ ]:
# scratch space
